In [ ]:
import os
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "0.99"

from calvin_env.envs.play_table_env import PlayTableSimEnv

In [ ]:
import calvin_env
from hydra import initialize, compose
import hydra

with initialize(config_path=f"../calvin_env/conf/"):
    cfg = compose(config_name="config_data_collection.yaml", overrides=["cameras=static_and_gripper"])
    cfg.env["use_egl"] = False
    cfg.env["show_gui"] = False
    cfg.env["use_vr"] = False
    cfg.env["use_scene_info"] = True
    print(cfg.env)

env = hydra.utils.instantiate(cfg.env)

In [ ]:
from openpi.policies import policy_config as _policy_config
from openpi.training import config as _config

import pathlib
def create_pi05(model_name, checkpoint_dir: str, assets_dir=None):
    vla_config = _config.get_config(model_name)
    checkpoint_dir = pathlib.Path(checkpoint_dir).resolve()

    norm_stats = None
    if assets_dir is not None:
        data_config = vla_config.data.create(vla_config.assets_dirs, vla_config.model)
        if data_config.asset_id is not None:
            from openpi.training import checkpoints as _checkpoints
            assets_path = pathlib.Path(assets_dir).resolve()
            norm_stats = _checkpoints.load_norm_stats(assets_path, data_config.asset_id)

    return _policy_config.create_trained_policy(
        vla_config,
        checkpoint_dir,
        norm_stats=norm_stats
    )

import openpi
openpi_root = pathlib.Path(openpi.__file__).resolve().parent.parent.parent
model_name = "pi05_calvin"
checkpoint_dir = openpi_root / "checkpoints" / "pi05_calvin" / "pi05_calvin" / "99999"
assets_dir = openpi_root / "assets" / "pi05_calvin"

model = create_pi05(model_name, checkpoint_dir, assets_dir)


In [ ]:
def prompt_from_obs(obs, prompt):
    return {
        "observation/image": obs['rgb_obs']['rgb_static'].astype(np.uint8),
        "observation/wrist_image": obs['rgb_obs']['rgb_gripper'].astype(np.uint8),
        "observation/state": obs['robot_obs'],
        "prompt": prompt,
        "thought": [prompt]
    }

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
observation = env.reset()

plt.figure(0)
plt.imshow(observation['rgb_obs']['rgb_static'])
plt.show()

In [ ]:
task = "pick up the red block"

prompt = prompt_from_obs(observation, task)
vla_output = model.infer(prompt)
print(vla_output)

action_idx = 0
#The observation is given as a dictionary with different values
# images = []
for i in range(40):
    # The action consists in a pose displacement (position and orientation)
    if action_idx == len(vla_output['actions']):
        prompt = prompt_from_obs(observation, task)
        vla_output = model.infer(prompt)
        action_idx = 0
    action = vla_output['actions'][action_idx].copy()
    print(action[-1])
    if action[-1] < 0:
        action[-1] = -1
    else:
        action[-1] = 1
    action_idx += 1
    observation, reward, done, info = env.step(action)
    rgb = np.array(env.render(mode="rgb_array"), dtype=np.uint8).copy()
    images.append(rgb)
#     plt.figure(i)
#     plt.clf()
#     plt.imshow(rgb)

In [ ]:
from calvin_agent.evaluation.multistep_sequences import get_sequences

In [ ]:
print(get_sequences(1))

In [ ]:
import mediapy
mediapy.write_video("out.mp4", images)